# End-to-end pretraining

This notebook creates a tiny temporary token dataset, runs the complete DeepSeek + MTP training path, evaluates the main model, and writes a resumable checkpoint.

In [1]:
import sys
from pathlib import Path
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'model').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import json
import tempfile
import numpy as np
import torch
from src.data import TokenBatcher
from src.model import DeepSeekConfig, DeepSeekWithMTP
from src.training import (MoELoadBalancer, create_optimizer, efficient_train_step, evaluate_main_model, save_training_checkpoint)


In [2]:
work_dir = Path(tempfile.mkdtemp(prefix='deepseek-module-27-'))
data_dir = work_dir / 'processed'
data_dir.mkdir()
rng = np.random.default_rng(27)
for split in ('train', 'val'):
    rng.integers(0, 64, size=2000, dtype=np.uint16).tofile(data_dir / f'{split}.bin')
(data_dir / 'metadata.json').write_text(json.dumps({'tokenizer_vocab_size': 64, 'dtype': '<u2'}))
config = DeepSeekConfig(vocab_size=64, max_seq_len=8, dim=32, n_layers=2, n_dense_layers=1, n_heads=4, q_lora_rank=8, kv_lora_rank=4, qk_nope_head_dim=8, qk_rope_head_dim=4, v_head_dim=8, inter_dim=64, moe_inter_dim=16, n_routed_experts=4, n_shared_experts=1, n_activated_experts=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepSeekWithMTP(config).to(device)
optimizer = create_optimizer(model)
train_batcher = TokenBatcher(data_dir / 'train.bin', seq_len=8, dtype='<u2', seed=27)
val_batcher = TokenBatcher(data_dir / 'val.bin', seq_len=8, dtype='<u2', seed=28)
print('device:', device, 'work directory:', work_dir)

device: cpu work directory: /var/folders/1x/sv570_bn4js_f82q2vj0f4qc0000gn/T/deepseek-module-27-_unlq_7r


In [3]:
with MoELoadBalancer(model, gamma=0.01) as balancer:
    for step in range(1, 3):
        batches = [train_batcher.next_batch(2).to(device) for _ in range(2)]
        metrics = efficient_train_step(model, optimizer, balancer, batches, step=step, total_steps=2, warmup_steps=1, use_bf16=False)
        validation = evaluate_main_model(model.main_model, val_batcher, batch_size=2, num_batches=1)
        balancer.clear()
        model.train()
        print(step, metrics['loss'], validation)
    save_training_checkpoint(work_dir / 'latest.pt', model=model, optimizer=optimizer, step=2, train_batcher=train_batcher, val_batcher=val_batcher, run_settings={'steps': 2, 'seq_len': 8}, metrics=metrics)
print('checkpoint:', work_dir / 'latest.pt')

1 5.577665567398071 {'val_loss': 4.307889461517334, 'perplexity': 74.28354510003287, 'evaluated_tokens': 16}
2 5.7188310623168945 {'val_loss': 4.42027473449707, 'perplexity': 83.1191179108114, 'evaluated_tokens': 16}
checkpoint: /var/folders/1x/sv570_bn4js_f82q2vj0f4qc0000gn/T/deepseek-module-27-_unlq_7r/latest.pt


The real entry point uses the same components with a persistent dataset and checkpoint path. The root Makefile provides the complete commands.